# Day 4

## Tokenizing with code

In [1]:
import tiktoken

encoding = tiktoken.encoding_for_model("gpt-4.1-mini")

tokens = encoding.encode("Hi my name is Ed and I like banoffee pie")

In [2]:
tokens

In [3]:
for token_id in tokens:
    token_text = encoding.decode([token_id])
    print(f"{token_id} = {token_text}")

12194 = Hi
922 =  my
1308 =  name
382 =  is
6117 =  Ed
326 =  and
357 =  I
1299 =  like
9171 =  ban
26458 = offee
5148 =  pie


In [4]:
encoding.decode([326])

# And another topic!

### The Illusion of "memory"

Many of you will know this already. But for those that don't -- this might be an "AHA" moment!

In [5]:
import os
from dotenv import load_dotenv

load_dotenv("../../.env", override=True)
openrouter_api_key = (os.getenv("OPENROUTER_API_KEY") or os.getenv("OPENAI_API_KEY") or "").strip().lstrip("=")

if not openrouter_api_key:
    print("No OpenRouter API key was found - please check your .env file!")
elif not openrouter_api_key.startswith("sk-or-v1-"):
    print("An API key was found, but it doesn't look like a valid OpenRouter key (should start with sk-or-v1-)")
else:
    print("OpenRouter API key found and looks good!")


OpenRouter API key found and looks good!


### You should be very comfortable with what the next cell is doing!

_I'm creating a new instance of the OpenAI Python Client library, a lightweight wrapper around making HTTP calls to an endpoint for calling the GPT LLM, or other LLM providers_

In [6]:
from openai import OpenAI

openai = OpenAI(
    api_key=openrouter_api_key,
    base_url="https://openrouter.ai/api/v1",
)


### A message to OpenAI is a list of dicts

In [7]:
messages = [
    {"role": "system", "content": "You are a helpful assistant"},
    {"role": "user", "content": "Hi! I'm Ed!"}
    ]

In [8]:
response = openai.chat.completions.create(model="openai/gpt-4o-mini", messages=messages, max_tokens=1000)
response.choices[0].message.content

'Hi Ed! How can I assist you today?'

### OK let's now ask a follow-up question

In [9]:
messages = [
    {"role": "system", "content": "You are a helpful assistant"},
    {"role": "user", "content": "What's my name?"}
    ]

In [10]:
response = openai.chat.completions.create(model="openai/gpt-4o-mini", messages=messages, max_tokens=1000)
response.choices[0].message.content

"I'm sorry, but I don't know your name. If you'd like, you can tell me your name!"

### Wait, wha??

We just told you!

What's going on??

Here's the thing: every call to an LLM is completely STATELESS. It's a totally new call, every single time. As AI engineers, it's OUR JOB to devise techniques to give the impression that the LLM has a "memory".

In [11]:
messages = [
    {"role": "system", "content": "You are a helpful assistant"},
    {"role": "user", "content": "Hi! I'm Ed!"},
    {"role": "assistant", "content": "Hi Ed! How can I assist you today?"},
    {"role": "user", "content": "What's my name?"}
    ]

In [12]:
response = openai.chat.completions.create(model="openai/gpt-4o-mini", messages=messages, max_tokens=1000)
response.choices[0].message.content

'Your name is Ed! How can I help you today, Ed?'

## To recap

With apologies if this is obvious to you - but it's still good to reinforce:

1. Every call to an LLM is stateless
2. We pass in the entire conversation so far in the input prompt, every time
3. This gives the illusion that the LLM has memory - it apparently keeps the context of the conversation
4. But this is a trick; it's a by-product of providing the entire conversation, every time
5. An LLM just predicts the most likely next tokens in the sequence; if that sequence contains "My name is Ed" and later "What's my name?" then it will predict.. Ed!

The ChatGPT product uses exactly this trick - every time you send a message, it's the entire conversation that gets passed in.

"Does that mean we have to pay extra each time for all the conversation so far"

For sure it does. And that's what we WANT. We want the LLM to predict the next tokens in the sequence, looking back on the entire conversation. We want that compute to happen, so we need to pay the electricity bill for it!

